# Development and testing for indeterminate population

## Script test

In [43]:
%load_ext autoreload
%autoreload 2

In [46]:
from src.visualization.sentencing import plotly_indeterminate_population

## Script development

In [1]:
# Importing libraries
import os

import chart_studio
import chart_studio.plotly as py
import pandas as pd
import plotly.graph_objs as go
import plotly.io as pio
from dotenv import find_dotenv, load_dotenv
from plotly.subplots import make_subplots

# Import prt_theme module
from src.visualization import prt_theme

# Loading environment variables
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

# Adding plotly credentials
chart_studio.tools.set_credentials_file(
    username=os.getenv("PLOTLY_USERNAME"), api_key=os.getenv("PLOTLY_API_KEY")
)

# Setting default Plotly template
pio.templates.default = "prt_template"

In [38]:
# Read in datasets
df = (
    pd.read_csv("data/processed/sentencing/indeterminate_population.csv", usecols=['year', 'indet_unreleased', 'indet_recalled'])
    .rename(columns={'indet_unreleased': 'Unreleased', 'indet_recalled': 'Recalled'})
    .melt(id_vars='year', value_vars=['Unreleased', 'Recalled'], var_name='custody_type')
)
df

,year,custody_type,value
0,2002,Unreleased,5146
1,2003,Unreleased,5419
2,2004,Unreleased,5595
3,2005,Unreleased,5882
4,2006,Unreleased,7274
5,2007,Unreleased,9481
6,2008,Unreleased,11383
7,2009,Unreleased,12182
8,2010,Unreleased,13134
9,2011,Unreleased,13644


In [ ]:
## Plotting
fig = go.Figure()

trace_list= []

for i in df["custody_type"].unique():
    df_type = df[df["custody_type"] == i]

    trace = go.Bar(
        x=df_type["year"],
        y=df_type["value"],
        name=str(df_type['custody_type'].iloc[0]),
        # customdata=df_type['year'],
        hovertemplate="%{y:,.0f}"
        )
    
    trace_list.append(trace)

fig.add_traces(trace_list)


fig.update_layout(
    barmode="stack",
    hovermode='x',
    xaxis_dtick=2,
    yaxis_tickformat= ",.0f",
    yaxis_automargin=True, #To avoid clipping of y-axis labels
    margin_pad=5,
    margin=dict(t=20, b=60, l=0, r=25),
)

## Chart annotations
annotations = []

# Add y-axis label annotation with placement based on dataframe column
prt_theme.add_annotation(
    annotations, "People in prison", annotation_type="y-axis"
)

# Adding annotations to layout
fig.update_layout(annotations=annotations)
fig.show()

In [23]:
fig.data[1]

Bar({
    'customdata': array([2022, 2022, 2022, 2022, 2022, 2022, 2022, 2022]),
    'hovertemplate': '<b>%{customdata}</b><br>%{x}: %{y} months<extra></extra>',
    'name': '2022',
    'text': array([120.8,  83.1,  59.2,  48.3,  45.3,  46.3,  30.1,   7.4]),
    'x': array(['Manslaughter', 'Aggravated<br>burglary', 'GBH with intent', 'Robbery',
                'Arson<br>endangering life',
                'Burglary in a<br>dwelling<br>(indictable)', 'Money laundering',
                'Knife possession'], dtype=object),
    'y': array([120.8,  83.1,  59.2,  48.3,  45.3,  46.3,  30.1,   7.4])
})

In [37]:
fig.layout

Layout({
    'annotations': [{'font': {'size': 14},
                     'showarrow': False,
                     'text': 'Average sentence length (months)',
                     'x': 0,
                     'xanchor': 'left',
                     'xref': 'paper',
                     'y': 1,
                     'yanchor': 'bottom',
                     'yref': 'paper'}],
    'barmode': 'group',
    'hovermode': 'closest',
    'images': [{'sizex': 0.5,
                'sizey': 0.5,
                'source': 'assets/up-arrow.svg',
                'x': 0.5,
                'xanchor': 'right',
                'xref': 'paper',
                'y': 0.5,
                'yanchor': 'bottom',
                'yref': 'paper'}],
    'margin': {'b': 60, 'l': 40, 'pad': 5, 'r': 25, 't': 20},
    'template': '...',
    'xaxis': {'tickangle': 0, 'ticks': 'inside'}
})

In [ ]:
for offence in fig.data[1]:
    fig.add_layout_image(
        dict(
            source=source,
            xref="x", yref="y",
            x=1, y=1.05,
            sizex=0.2, sizey=0.2,
            xanchor="right", yanchor="bottom"
        )
        )